In [ ]:
import pickle
import numpy as np
from scipy.sparse import csr_matrix, save_npz
from surprise import Dataset, Reader
from surprise.model_selection import GridSearchCV
from loaders import load_ratings
from constants import Constant as C

from models import UserBased_tuned 

# 1. Loading the data
df = load_ratings()
reader = Reader(rating_scale=C.RATINGS_SCALE)
data = Dataset.load_from_df(df[[C.USER_ID_COL, C.ITEM_ID_COL, C.RATING_COL]], reader)

# 2. Gridsearch
param_grid = {
    'k': [20, 30, 40, 50],
    'min_k': [3, 5, 7],
    'sim_options': {'name': ['msd', 'jacard', 'cosine_jaccard'], 'min_support': [3, 5]}
}

print("Tuning parameters in progress...")
gs = GridSearchCV(UserBased_tuned, param_grid, measures=['rmse', 'mae'], cv=5, n_jobs=-1)
gs.fit(data)

print(f"Meilleurs paramètres trouvés : {gs.best_params['rmse']}")

# =========================================================================
# 3. Final training with the best parameters
# =========================================================================
best_k = gs.best_params['rmse']['k']
best_min_k = gs.best_params['rmse']['min_k']

best_sim_name = gs.best_params['rmse']['sim_options']['name']
best_min_support = gs.best_params['rmse']['sim_options']['min_support']

print(f"\n Final configuration for the HTML site :")
print(f"-> Metric : {best_sim_name.upper()} (k={best_k}, min_k={best_min_k}, min_support={best_min_support})")

trainset = data.build_full_trainset()

ub_algo = UserBased_tuned(
    k=best_k, 
    min_k=best_min_k, 
    sim_options={'name': best_sim_name, 'min_support': best_min_support}
)
ub_algo.fit(trainset)

# =========================================================================
# 4. Exporting the artifacts
# =========================================================================
ts = ub_algo.trainset
rows, cols, vals = [], [], []
for u in range(ts.n_users):
    for iid, r in ts.ur[u]:
        rows.append(u); cols.append(iid); vals.append(r)
R = csr_matrix((vals, (rows, cols)), shape=(ts.n_users, ts.n_items))

save_npz("backend/artifacts/rating_matrix.npz", R)
means = np.array([np.mean([r for _, r in ts.ur[u]]) for u in range(ts.n_users)])
np.save("backend/artifacts/user_means.npy", means)

with open("backend/artifacts/userbased_model.pkl", "wb") as f:
    pickle.dump(ub_algo, f)

print(f"Successful save of the configured artifacts with the metric : {best_sim_name}")

In [ ]:
# Display in a nice dataframe
import pandas as pd
df_results = pd.DataFrame(gs.cv_results)

df_results.to_csv("user_based_tuning_results.csv", index=False)